# Phase 3 — orchestrator comparison study

**Backfill for Day 17** (run date 2026-05-20). Reads the canonical artifacts already on disk and re-renders the comparison tables + charts in notebook form. No new experiments are run here — the harness is the source of truth.

**Inputs read:**
* `results/phase3_orchestrator_results.json` — 800 per-scenario rows (200 scenarios × 4 strategies) plus per-strategy aggregates with correctness, latency, cost, confusion matrix, per-tenant + per-event-type rollup, static rubric scores.
* `benchmarks/data/orchestrator/manifest.json` — dataset shape (3 tenants, decision-class counts, difficulty counts).
* `results/phase3_day18_consolidated.json` — the Day-18 wrap view that names champions for both Phase-3 halves.

**LLM mode of the source numbers:** mock. Naive applies a tenant-agnostic event-type heuristic (small-LLM failure mode); LLM-as-judge reads the policy table directly (perfect-LLM emulation). Both behaviours are documented in their engine modules. Phase 5 / Day 28 re-runs against a real LLM provider.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = ROOT / "results"
DATA = ROOT / "benchmarks" / "data" / "orchestrator"

results = json.loads((RESULTS / "phase3_orchestrator_results.json").read_text(encoding="utf-8"))
manifest = json.loads((DATA / "manifest.json").read_text(encoding="utf-8"))
consolidated = json.loads((RESULTS / "phase3_day18_consolidated.json").read_text(encoding="utf-8"))

summary = results["summary"]
strategies = ["declarative", "python_rules", "naive_llm", "llm_judge"]
print(f"scenarios: {results['n_scenarios']}; strategies: {strategies}")
print(f"tenants: {manifest['tenants']}")
print(f"decision_counts: {manifest['decision_counts']}")
print(f"champion: {consolidated['orchestrator']['champion']}")


## 1. Headline per-strategy table

Correctness, p50 / p95 latency, cost per 100 decisions, and the static auditability + maintainability rubric scores.


In [ ]:
header = f"{'strategy':<14}{'corr':>8}{'p50µs':>8}{'p95µs':>8}{'$/100':>10}{'aud':>5}{'mnt':>5}"
print(header)
print('-' * len(header))
for s in strategies:
    e = summary[s]
    print(
        f"{s:<14}"
        f"{e['correctness']:>8.3f}"
        f"{e['decision_latency_us_p50']:>8.2f}"
        f"{e['decision_latency_us_p95']:>8.2f}"
        f"{'$' + format(e['estimated_cost_per_100_decisions_usd'], '.3f'):>10}"
        f"{e['auditability']:>5}"
        f"{e['maintainability']:>5}"
    )


## 2. Correctness — where do the engines that miss, miss?

Declarative / python_rules / llm_judge all score 1.000. Naive falls to 0.540 — and the failure pattern is structured, not noisy: it misses every `reject` scenario, fails worst on the tenant with the most idiosyncratic policy, and degrades in proportion to how much tenant policy varies by event type.


In [ ]:
correctness = [summary[s]['correctness'] for s in strategies]
colors = ['#2ca02c', '#17becf', '#d62728', '#ff7f0e']
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(strategies, correctness, color=colors)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Correctness (fraction of 200 scenarios)')
ax.set_title('Orchestrator policy engines — correctness on 200 scenarios')
ax.axhline(1.0, color='#888', linewidth=0.5, linestyle='--')
for bar, val in zip(bars, correctness):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.015, f'{val:.3f}', ha='center')
fig.tight_layout()
out = RESULTS / 'phase3_orchestrator_correctness.png'
fig.savefig(out, dpi=120)
print(f'saved {out}')


In [ ]:
naive = summary['naive_llm']
print('per-tenant correctness (naive):')
for t, v in sorted(naive['per_tenant_correctness'].items()):
    print(f'  {t:<32}{v:.3f}')
print('\nper-event-type correctness (naive):')
for t, v in sorted(naive['per_event_type_correctness'].items()):
    print(f'  {t:<24}{v:.3f}')
print('\nper-expected-decision correctness (naive):')
for t, v in sorted(naive['per_expected_decision_correctness'].items()):
    print(f'  {t:<24}{v:.3f}')


## 3. Cost / latency frontier

LLM-engines (naive, llm_judge) pay both $$ and µs; rule-based engines (declarative, python_rules) pay neither.


In [ ]:
cost = [summary[s]['estimated_cost_per_100_decisions_usd'] for s in strategies]
p50 = [summary[s]['decision_latency_us_p50'] for s in strategies]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].bar(strategies, cost, color='#9467bd')
axes[0].set_ylabel('USD per 100 decisions')
axes[0].set_title('Cost per 100 decisions')
for i, v in enumerate(cost):
    axes[0].text(i, v + (max(cost)*0.02 if max(cost) else 0.005), f'${v:.3f}', ha='center')
axes[1].bar(strategies, p50, color='#8c564b')
axes[1].set_ylabel('p50 latency (µs)')
axes[1].set_title('p50 decision latency')
for i, v in enumerate(p50):
    axes[1].text(i, v + max(p50)*0.02, f'{v:.1f}µs', ha='center')
fig.tight_layout()
out = RESULTS / 'phase3_orchestrator_cost_latency.png'
fig.savefig(out, dpi=120)
print(f'saved {out}')


## 4. Findings

1. **Declarative YAML wins outright on this dataset.** 100% correctness, sub-microsecond p50, $0 marginal cost, 5/5 on both rubric scores. The four-strategy bake-off was supposed to find a champion; the Day-10 incumbent is unbeaten.
2. **Naive LLM ('paste policy into prompt') drops to 54% correctness AND misses every `reject` scenario (5/5 wrong) — silently routing high-risk actions to auto-execute.** This is exactly the bug class regulators audit for; the comparison study's headline finding.
3. **LLM-as-judge ties on correctness but is ~11× slower and costs ~$0.11/100 decisions.** For clearly-stated policy tables, promoting decisions to an LLM judge buys nothing — the information is already in the dict. LLM-as-judge's slice (if any) is ambiguous policies; Day 28 will test that.
4. **Naive fails worst on the tenant with the most idiosyncratic policy** (Jefferson Credit: 0.478 vs Acme 0.597 vs Globetrek 0.546). Tenant-blind heuristics fail in proportion to how much tenants diverge from each other — predictable, measurable, and exactly what the multi-tenant study should surface.

## Next analysis to land here

* **Day 28 (Phase 5)** — real-LLM re-run replaces the mock-mode naive heuristic and the perfect-LLM judge emulation with actual model output; same harness, same dataset, swap the client.
